# DALI — Entraînement **100 % Google Colab** (sans Drive)

Ce notebook **ne nécessite pas** Google Drive : les fichiers `prepare_training_data.py` et `train_improved_model.py` ont été **embarqués** dans le notebook au moment de sa génération sur ton PC.

### Sur ton PC (une fois)
Dans le dossier `modele_moteur_ia_inspect` :
```bash
python build_colab_notebook.py
```

### Sur Colab
1. **Fichier → Importer une note** → choisir `Colab_entrainement_AI4I.ipynb`.
2. **Exécution → Tout exécuter** (ou cellule par cellule, de haut en bas).
3. À la fin : téléchargement du **ZIP** des modèles (`best_model_v3.keras`, `scaler.pkl`, `metadata.json`, …).

### GPU (recommandé)
**Exécution → Modifier le type d’exécution → T4 GPU** (accélère TensorFlow).

### Données
Le CSV **AI4I 2020** est téléchargé depuis **UCI** (équivalent au jeu Kaggle).


## 1) Dépendances


In [ ]:
%pip install -q tensorflow pandas scikit-learn matplotlib joblib


## 2) Télécharger le dataset AI4I (UCI)


In [ ]:
import urllib.request
from pathlib import Path

Path("/content/data").mkdir(parents=True, exist_ok=True)
RAW = "/content/data/ai4i2020.csv"
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
print("Téléchargement...", url)
urllib.request.urlretrieve(url, RAW)

import pandas as pd
df = pd.read_csv(RAW)
print("Shape:", df.shape)
print(df.head(2))


## 3) Écrire les scripts du projet (embarqués)

Les deux `.py` sont recréés sous `/content/scripts/` à partir du notebook.


In [ ]:
import base64
from pathlib import Path

Path("/content/scripts").mkdir(parents=True, exist_ok=True)

prep_b64 = "aW1wb3J0IGFyZ3BhcnNlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpkZWYgbm9ybWFsaXplX2FpNGlfY29sdW1ucyhkZjogcGQuRGF0YUZyYW1lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiIKICAgIEhhcm1vbmlzZSBsZXMgZW4tdMOqdGVzIGVudHJlIGxlIENTViBVQ0kgb2ZmaWNpZWwgZXQgY2VydGFpbmVzIHZhcmlhbnRlcyBLYWdnbGUKICAgIChleC4gY29sb25uZXMgc2FucyBbS10gLyBbcnBtXSkuCiAgICAiIiIKICAgIGRmID0gZGYuY29weSgpCiAgICBkZi5jb2x1bW5zID0gW3N0cihjKS5zdHJpcCgpIGZvciBjIGluIGRmLmNvbHVtbnNdCgogICAgYWxpYXNlcyA9IHsKICAgICAgICAiQWlyIHRlbXBlcmF0dXJlIjogIkFpciB0ZW1wZXJhdHVyZSBbS10iLAogICAgICAgICJQcm9jZXNzIHRlbXBlcmF0dXJlIjogIlByb2Nlc3MgdGVtcGVyYXR1cmUgW0tdIiwKICAgICAgICAiUm90YXRpb25hbCBzcGVlZCI6ICJSb3RhdGlvbmFsIHNwZWVkIFtycG1dIiwKICAgICAgICAiVG9ycXVlIjogIlRvcnF1ZSBbTm1dIiwKICAgICAgICAiVG9vbCB3ZWFyIjogIlRvb2wgd2VhciBbbWluXSIsCiAgICB9CiAgICBmb3Igc2hvcnQsIGZ1bGwgaW4gYWxpYXNlcy5pdGVtcygpOgogICAgICAgIGlmIHNob3J0IGluIGRmLmNvbHVtbnMgYW5kIGZ1bGwgbm90IGluIGRmLmNvbHVtbnM6CiAgICAgICAgICAgIGRmID0gZGYucmVuYW1lKGNvbHVtbnM9e3Nob3J0OiBmdWxsfSkKCiAgICAjIFNjw6luYXJpbyA6IGTDqXJpdmVyIGRlcHVpcyBsZXMgZmxhZ3MgQUk0SSBzaSBwYXMgZGUgY29sb25uZSBGYWlsdXJlIFR5cGUKICAgIGlmICJGYWlsdXJlIFR5cGUiIG5vdCBpbiBkZi5jb2x1bW5zIGFuZCAiSERGIiBpbiBkZi5jb2x1bW5zOgogICAgICAgIGRlZiBfZmFpbHVyZV90eXBlKHI6IHBkLlNlcmllcykgLT4gc3RyOgogICAgICAgICAgICBpZiBpbnQoci5nZXQoIkhERiIsIDApIG9yIDApID09IDE6CiAgICAgICAgICAgICAgICByZXR1cm4gIkhlYXQgRGlzc2lwYXRpb24gRmFpbHVyZSIKICAgICAgICAgICAgaWYgaW50KHIuZ2V0KCJQV0YiLCAwKSBvciAwKSA9PSAxOgogICAgICAgICAgICAgICAgcmV0dXJuICJQb3dlciBGYWlsdXJlIgogICAgICAgICAgICBpZiBpbnQoci5nZXQoIk9TRiIsIDApIG9yIDApID09IDE6CiAgICAgICAgICAgICAgICByZXR1cm4gIk92ZXJzdHJhaW4gRmFpbHVyZSIKICAgICAgICAgICAgaWYgaW50KHIuZ2V0KCJUV0YiLCAwKSBvciAwKSA9PSAxOgogICAgICAgICAgICAgICAgcmV0dXJuICJUb29sIFdlYXIgRmFpbHVyZSIKICAgICAgICAgICAgaWYgaW50KHIuZ2V0KCJSTkYiLCAwKSBvciAwKSA9PSAxOgogICAgICAgICAgICAgICAgcmV0dXJuICJSYW5kb20gRmFpbHVyZXMiCiAgICAgICAgICAgIHJldHVybiAiTm8gRmFpbHVyZSIKCiAgICAgICAgZGZbIkZhaWx1cmUgVHlwZSJdID0gZGYuYXBwbHkoX2ZhaWx1cmVfdHlwZSwgYXhpcz0xKQoKICAgIHJldHVybiBkZgoKCmRlZiBtYXBfc2NlbmFyaW8ocm93OiBwZC5TZXJpZXMpIC0+IHN0cjoKICAgIGZ0ID0gc3RyKHJvdy5nZXQoIkZhaWx1cmUgVHlwZSIsICJObyBGYWlsdXJlIikpLnN0cmlwKCkubG93ZXIoKQogICAgaWYgImhlYXQiIGluIGZ0OgogICAgICAgIHJldHVybiAiU1VSQ0hBVUZGRSIKICAgIGlmICJwb3dlciIgaW4gZnQ6CiAgICAgICAgcmV0dXJuICJTVVJDSEFSR0UiCiAgICBpZiAib3ZlcnN0cmFpbiIgaW4gZnQ6CiAgICAgICAgcmV0dXJuICJVU1VSRV9HRU5FUkFMRSIKICAgIGlmICJ0b29sIHdlYXIiIGluIGZ0OgogICAgICAgIHJldHVybiAiUk9VTEVNRU5UIgogICAgaWYgInJhbmRvbSIgaW4gZnQ6CiAgICAgICAgcmV0dXJuICJFTEVDVFJJUVVFIgogICAgcmV0dXJuICJOT1JNQUwiCgoKZGVmIGJ1aWxkX2Zyb21fYWk0aShkZjogcGQuRGF0YUZyYW1lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAjIE1hcCBBSTRJIGNvbHVtbnMgdG8gZXhwZWN0ZWQgdHJhaW5pbmcgc2NoZW1hLgogICAgb3V0ID0gcGQuRGF0YUZyYW1lKCkKICAgIG91dFsidGVtcGVyYXR1cmUiXSA9ICgoZGZbIkFpciB0ZW1wZXJhdHVyZSBbS10iXSArIGRmWyJQcm9jZXNzIHRlbXBlcmF0dXJlIFtLXSJdKSAvIDIuMCkuYXN0eXBlKGZsb2F0KQogICAgb3V0WyJwcmVzc2lvbiJdID0gKGRmWyJUb3JxdWUgW05tXSJdIC8gbnAubWF4aW11bShkZlsiUm90YXRpb25hbCBzcGVlZCBbcnBtXSJdLCAxKSkuYXN0eXBlKGZsb2F0KQogICAgb3V0WyJwdWlzc2FuY2UiXSA9IChkZlsiVG9ycXVlIFtObV0iXSAqIGRmWyJSb3RhdGlvbmFsIHNwZWVkIFtycG1dIl0pLmFzdHlwZShmbG9hdCkKICAgIG91dFsidmlicmF0aW9uIl0gPSAoZGZbIlJvdGF0aW9uYWwgc3BlZWQgW3JwbV0iXSAvIDEwMDAuMCkuYXN0eXBlKGZsb2F0KQogICAgb3V0WyJwcmVzZW5jZSJdID0gMS4wCiAgICBvdXRbIm1hZ25ldGlxdWUiXSA9IGRmWyJUeXBlIl0ubWFwKHsiTCI6IDAuMywgIk0iOiAwLjYsICJIIjogMC45fSkuZmlsbG5hKDAuNSkuYXN0eXBlKGZsb2F0KQogICAgb3V0WyJpbmZyYXJvdWdlIl0gPSBkZlsiUHJvY2VzcyB0ZW1wZXJhdHVyZSBbS10iXS5hc3R5cGUoZmxvYXQpCgogICAgb3V0WyJ0eXBlX21vdGV1ciJdID0gZGZbIlR5cGUiXS5tYXAoeyJMIjogIkVMX1MiLCAiTSI6ICJFTF9NIiwgIkgiOiAiRUxfTCJ9KS5maWxsbmEoIkVMX00iKQogICAgdGFyZ2V0X2NvbCA9ICJUYXJnZXQiIGlmICJUYXJnZXQiIGluIGRmLmNvbHVtbnMgZWxzZSAiTWFjaGluZSBmYWlsdXJlIgogICAgb3V0WyJwYW5uZSJdID0gZGZbdGFyZ2V0X2NvbF0uYXN0eXBlKGludCkKICAgIG91dFsiYW5vbWFsaWUiXSA9IGRmW3RhcmdldF9jb2xdLmFzdHlwZShpbnQpCiAgICBvdXRbInNjZW5hcmlvIl0gPSBkZi5hcHBseShtYXBfc2NlbmFyaW8sIGF4aXM9MSkKCiAgICAjIFByb3h5IFJVTCBmcm9tIHRvb2wgd2VhcjogbGFyZ2VyIHdlYXIgLT4gbG93ZXIgcmVtYWluaW5nIGxpZmUuCiAgICB3ZWFyID0gZGZbIlRvb2wgd2VhciBbbWluXSJdLmFzdHlwZShmbG9hdCkKICAgIG1heF93ZWFyID0gbWF4KGZsb2F0KHdlYXIubWF4KCkpLCAxLjApCiAgICBvdXRbInJ1bCJdID0gKG1heF93ZWFyIC0gd2VhcikuY2xpcChsb3dlcj0wLjApCiAgICByZXR1cm4gb3V0CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlByZXBhcmUgdW5pZmllZCB0cmFpbmluZyBkYXRhc2V0IGZvciBpbXByb3ZlZCBtb2RlbC4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1haTRpIiwgbmFyZ3M9IisiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJMaXN0IG9mIEFJNEktc3R5bGUgQ1NWIGZpbGVzLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9Ik91dHB1dCBDU1YgcGF0aC4iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBmcmFtZXMgPSBbXQogICAgZm9yIGNzdl9wYXRoIGluIGFyZ3MuYWk0aToKICAgICAgICBwID0gUGF0aChjc3ZfcGF0aCkKICAgICAgICBkZiA9IHBkLnJlYWRfY3N2KHApCiAgICAgICAgZGYgPSBub3JtYWxpemVfYWk0aV9jb2x1bW5zKGRmKQogICAgICAgIGZyYW1lcy5hcHBlbmQoYnVpbGRfZnJvbV9haTRpKGRmKSkKCiAgICBkYXRhID0gcGQuY29uY2F0KGZyYW1lcywgaWdub3JlX2luZGV4PVRydWUpLmRyb3BfZHVwbGljYXRlcygpCiAgICBQYXRoKGFyZ3Mub3V0KS5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZGF0YS50b19jc3YoYXJncy5vdXQsIGluZGV4PUZhbHNlKQogICAgcHJpbnQoZiJTYXZlZCBwcmVwYXJlZCBkYXRhc2V0OiB7YXJncy5vdXR9IikKICAgIHByaW50KGYiU2hhcGU6IHtkYXRhLnNoYXBlfSIpCiAgICBwcmludCgiU2NlbmFyaW9zOiIsIGRhdGFbInNjZW5hcmlvIl0udmFsdWVfY291bnRzKCkudG9fZGljdCgpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"
train_b64 = "..."

open("/content/scripts/prepare_training_data.py", "wb").write(base64.b64decode(prep_b64))
open("/content/scripts/train_improved_model.py", "wb").write(base64.b64decode(train_b64))
print("OK : /content/scripts/prepare_training_data.py")
print("OK : /content/scripts/train_improved_model.py")


## 4) Préparer le CSV d’entraînement


In [ ]:
import os
from pathlib import Path

Path("/content/dali_out").mkdir(parents=True, exist_ok=True)
os.chdir("/content/scripts")
!python prepare_training_data.py --ai4i /content/data/ai4i2020.csv --out /content/dali_out/train_prepared.csv


## 5) Entraîner le modèle LSTM (réduire `--epochs` pour un test rapide)


In [ ]:
import os
os.chdir("/content/scripts")
!python train_improved_model.py --data /content/dali_out/train_prepared.csv --out-dir /content/dali_out/models_colab --epochs 45 --batch-size 128


## 6) Télécharger les artefacts sur ton PC

Dézippe le fichier puis copie le dossier vers `modele_moteur_ia_inspect/models_v3_lstm` ou configure `MODEL_ARTIFACTS_DIR` pour `inference_api`.


In [ ]:
!ls -la /content/dali_out/models_colab
!zip -r /content/dali_models_colab.zip /content/dali_out/models_colab
from google.colab import files
files.download("/content/dali_models_colab.zip")


---

### Optionnel : Google Drive

Si tu préfères garder les `.py` sur Drive au lieu du notebook embarqué, utilise une ancienne version du notebook ou monte Drive et lance les mêmes commandes `python` depuis ton dossier Drive.
